In [ ]:
# ===========================================================================
# 30-feature-engineering.ipynb  |  Datathon 2026 - The Gridbreaker
# v3 adds (feedback-fe): peak May, CR proxy, premium margin gap, promo x AOV, 2022 recovery, organic decomposition
# v2: promo x post-2019, traffic nonlinear + imputation flag, returns/reviews (lag-safe)
# Input : data/*.csv
# Output: data/features/train_features.parquet
#         data/features/test_features.parquet
# Leakage rule: ALL features at date t use ONLY data from dates <= t-1
# ===========================================================================
import os, sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)

DATA_DIR  = '../data/data_cleaned'
FEAT_DIR  = '../data/features'
CHART_DIR = '../reports/charts/fe'
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(CHART_DIR, exist_ok=True)

TRAIN_START = pd.Timestamp('2012-07-04')
TRAIN_END   = pd.Timestamp('2022-12-31')
TEST_START  = pd.Timestamp('2023-01-01')
TEST_END    = pd.Timestamp('2024-07-01')

print(f'Train: {TRAIN_START.date()} -> {TRAIN_END.date()}')
print(f'Test : {TEST_START.date()} -> {TEST_END.date()}')


Train: 2012-07-04 -> 2022-12-31
Test : 2023-01-01 -> 2024-07-01


In [ ]:
# --- Cell 1: Load Data + Build Date Spine ---
from src.data_loader import (
    load_sales, load_promotions, load_web_traffic, load_inventory,
    load_orders, load_order_items, load_products,
    load_returns, load_reviews,
)

sales       = load_sales()
promotions  = load_promotions()
web_traffic = load_web_traffic()
inventory   = load_inventory()
orders      = load_orders()
order_items = load_order_items()
products    = load_products()
returns_tbl = load_returns()
reviews_tbl = load_reviews()

# Test skeleton: sample_submission has Date,Revenue,COGS -- keep Date ONLY (prevent leakage)
sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv', parse_dates=['Date'])

# Build date spine: every day TRAIN_START -> TEST_END
all_dates = pd.DataFrame({'Date': pd.date_range(TRAIN_START, TEST_END, freq='D')})

# Merge train targets (Revenue,COGS will be NaN for test dates -- expected and correct)
df = all_dates.merge(sales[['Date', 'Revenue', 'COGS']], on='Date', how='left')
df['is_train'] = df['Date'] <= TRAIN_END
df = df.sort_values('Date').reset_index(drop=True)

assert df.loc[~df['is_train'], 'Revenue'].isna().all(), '[FAIL] Test rows should have NaN Revenue'
print(f'Spine: {len(df)} rows | Train: {df["is_train"].sum()} | Test: {(~df["is_train"]).sum()}')
print(f'Test Revenue all NaN: [OK]')
for name, dd in [('sales', sales), ('promotions', promotions),
                  ('web_traffic', web_traffic), ('inventory', inventory),
                  ('orders', orders), ('products', products),
                  ('returns', returns_tbl), ('reviews', reviews_tbl)]:
    print(f'  {name:<14}: {len(dd):>7,} rows  {dd.shape[1]} cols')


Spine: 4381 rows | Train: 3833 | Test: 548
Test Revenue all NaN: [OK]
  sales         :   3,833 rows  3 cols
  promotions    :      50 rows  10 cols
  web_traffic   :   3,652 rows  7 cols
  inventory     :  60,247 rows  17 cols
  orders        : 646,945 rows  8 cols
  products      :   2,412 rows  8 cols


In [ ]:
# --- Cell 2: Calendar Features (no leakage -- deterministic from Date) ---
df['dayofweek']      = df['Date'].dt.dayofweek          # 0=Mon, 6=Sun
df['dayofmonth']     = df['Date'].dt.day
df['dayofyear']      = df['Date'].dt.dayofyear
df['month']          = df['Date'].dt.month
df['quarter']        = df['Date'].dt.quarter
df['year']           = df['Date'].dt.year
df['weekofyear']     = df['Date'].dt.isocalendar().week.astype(int)
df['is_weekend']     = (df['dayofweek'] >= 5).astype(int)
df['is_month_end']   = df['Date'].dt.is_month_end.astype(int)
df['is_month_start'] = df['Date'].dt.is_month_start.astype(int)
df['is_quarter_end'] = df['Date'].dt.is_quarter_end.astype(int)

# Fourier encoding for monthly and day-of-week seasonality
for k in [1, 2, 3]:
    df[f'sin_month_{k}'] = np.sin(2 * np.pi * k * df['month'] / 12)
    df[f'cos_month_{k}'] = np.cos(2 * np.pi * k * df['month'] / 12)
for k in [1, 2]:
    df[f'sin_dow_{k}'] = np.sin(2 * np.pi * k * df['dayofweek'] / 7)
    df[f'cos_dow_{k}'] = np.cos(2 * np.pi * k * df['dayofweek'] / 7)

# Tet Nguyen Dan (Lunar New Year) proximity -- dates 2013-2025 in Gregorian
TET_DATES = pd.to_datetime([
    '2013-02-10', '2014-01-31', '2015-02-19', '2016-02-08',
    '2017-01-28', '2018-02-16', '2019-02-05', '2020-01-25',
    '2021-02-12', '2022-02-01', '2023-01-22', '2024-02-10', '2025-01-29',
])

def tet_proximity(date, tet_list, window=30):
    diffs = [(date - t).days for t in tet_list]
    closest = min(diffs, key=abs)
    return closest if abs(closest) <= window else 0

df['tet_proximity_days'] = df['Date'].apply(lambda d: tet_proximity(d, TET_DATES))
df['is_tet_window'] = (df['tet_proximity_days'] != 0).astype(int)
df['is_pre_tet']    = (df['tet_proximity_days'] < 0).astype(int)
df['is_post_tet']   = (df['tet_proximity_days'] > 0).astype(int)

# Vietnamese fixed public holidays (solar calendar)
FIXED_HOL = {(1, 1), (4, 30), (5, 1), (9, 2)}
df['is_public_holiday']  = df['Date'].apply(lambda d: int((d.month, d.day) in FIXED_HOL))
df['is_year_end_season'] = df['month'].isin([11, 12]).astype(int)
df['is_back_to_school']  = df['month'].isin([8, 9]).astype(int)

# Post-2019 structural break (EDA: conversion rate dropped sharply after 2019)
df['post_2019_regime'] = (df['year'] >= 2019).astype(int)

# feedback-fe: May peak season
df['is_peak_may'] = (df['month'] == 5).astype(int)

# feedback-fe: CR recovery / positive swing in 2022 (calendar signal)
df['is_recovery_year_2022'] = (df['year'] == 2022).astype(int)

# Linear trend for long-horizon extrapolation into 2023-2024
df['days_since_start'] = (df['Date'] - TRAIN_START).dt.days
df['year_index'] = df['year'] - 2012  # 0=2012, 10=2022, 11+=extrapolation zone

cal_cols = [c for c in df.columns if c not in ['Date', 'Revenue', 'COGS', 'is_train']]
print(f'Calendar features: {len(cal_cols)}')


Calendar features: 31


In [ ]:
# --- Cell 3: Lag & Rolling Features on Revenue/COGS ---
# LEAKAGE GUARD: ALL shifts use n >= 1. Never shift(0) on Revenue/COGS.
df = df.sort_values('Date').reset_index(drop=True)
rev  = df['Revenue'].copy()
cogs = df['COGS'].copy()

# Revenue lags
for lag in [1, 2, 3, 7, 14, 21, 28, 60, 90, 180, 365]:
    df[f'rev_lag_{lag}'] = rev.shift(lag)

# Rolling statistics (shift 1 before rolling to prevent same-day inclusion)
for window in [7, 14, 28, 60, 90]:
    shifted = rev.shift(1)
    df[f'rev_roll_mean_{window}'] = shifted.rolling(window, min_periods=max(1, window // 2)).mean()
    df[f'rev_roll_std_{window}']  = shifted.rolling(window, min_periods=max(1, window // 2)).std()
    df[f'rev_roll_min_{window}']  = shifted.rolling(window, min_periods=max(1, window // 2)).min()
    df[f'rev_roll_max_{window}']  = shifted.rolling(window, min_periods=max(1, window // 2)).max()

df['rev_expanding_mean'] = rev.shift(1).expanding().mean()

# Momentum / relative features
df['rev_ratio_7_28']  = df['rev_lag_7']  / (df['rev_lag_28']  + 1e-9)
df['rev_ratio_28_90'] = df['rev_lag_28'] / (df['rev_lag_90']  + 1e-9)
df['rev_ratio_7_365'] = df['rev_lag_7']  / (df['rev_lag_365'] + 1e-9)
df['rev_wow'] = rev.shift(1) / (rev.shift(8)   + 1e-9) - 1  # week-over-week
df['rev_yoy'] = rev.shift(1) / (rev.shift(366) + 1e-9) - 1  # year-over-year

# COGS lags (parallel target forecasting)
for lag in [1, 7, 28, 365]:
    df[f'cogs_lag_{lag}'] = cogs.shift(lag)
df['cogs_roll_mean_28'] = cogs.shift(1).rolling(28, min_periods=14).mean()

# ─── Horizon-safe lags ────────────────────────────────────────────────────────
# lag_730 references 2021 data, which is always within the training period.
# These are the only Revenue lags guaranteed to be non-NaN for ALL 548 test days.
for lag in [729, 730, 731]:
    df[f'rev_lag_{lag}'] = rev.shift(lag)
df['cogs_lag_730'] = cogs.shift(730)

# Year-aligned rolling anchored at 365 days
# shift(365) puts us into 2022 data for 2023 test dates -- safe for 365/548 test days
for window in [7, 14, 28]:
    anchored_1y = rev.shift(365)
    df[f'rev_yoy_roll_mean_{window}'] = anchored_1y.rolling(window, min_periods=window // 2).mean()
    df[f'rev_yoy_roll_std_{window}']  = anchored_1y.rolling(window, min_periods=window // 2).std()

# 2-year-aligned rolling anchored at 730 days
# shift(730) puts us into 2021 data -- safe for ALL 548 test days including 2024
for window in [7, 14, 28]:
    anchored_2y = rev.shift(730)
    df[f'rev_2yoy_roll_mean_{window}'] = anchored_2y.rolling(window, min_periods=window // 2).mean()

# YoY ratio between 1-year-ago and 2-year-ago (encodes annual growth signal)
df['rev_ratio_365_730'] = df['rev_lag_365'] / (df['rev_lag_730'] + 1e-9)

# Annual growth rate (smoothed) -- key feature for extrapolating into test period
yoy_raw = (rev / rev.shift(365) - 1).shift(1)   # yesterday's YoY growth rate
df['rev_yoy_growth_90d']  = yoy_raw.rolling(90,  min_periods=30).mean()
df['rev_yoy_growth_180d'] = yoy_raw.rolling(180, min_periods=60).mean()

# ─── Same-weekday aligned lags ────────────────────────────────────────────────
# 364 = 52 x 7: always maps the same day-of-week (Monday->Monday, Sunday->Sunday).
# rev_lag_365 drifts weekday by 1 each year; lag_364 is seasonally cleaner.
# lag_364 is safe for 364/548 test days (2023 portion); lag_728 for ALL 548.
df['rev_lag_364'] = rev.shift(364)
df['rev_lag_357'] = rev.shift(357)   # 1 week before same-weekday last year
df['rev_lag_371'] = rev.shift(371)   # 1 week after same-weekday last year
df['rev_same_week_1y_mean'] = (df['rev_lag_357'] + df['rev_lag_364'] + df['rev_lag_371']) / 3

# 2-year same-weekday (728 = 2 x 364), safe for ALL test days including 2024
df['rev_lag_728'] = rev.shift(728)
df['rev_lag_721'] = rev.shift(721)
df['rev_lag_735'] = rev.shift(735)
df['rev_same_week_2y_mean'] = (df['rev_lag_721'] + df['rev_lag_728'] + df['rev_lag_735']) / 3

lag_cols = [c for c in df.columns if any(k in c for k in
            ['rev_lag', 'rev_roll', 'rev_ratio', 'rev_wow', 'rev_yoy', 'rev_exp',
             'cogs_lag', 'cogs_roll', 'rev_2yoy', 'rev_yoy_roll', 'rev_yoy_growth',
             'rev_same_week'])]
print(f'Lag/rolling features (total): {len(lag_cols)}')
print('NaN in train (top 5):')
print(df[df['is_train']].isna().sum().sort_values(ascending=False).head())

Lag/rolling features (total): 66
NaN in train (top 5):
rev_2yoy_roll_mean_28    743
rev_2yoy_roll_mean_14    736
rev_same_week_2y_mean    735
rev_lag_735              735
rev_2yoy_roll_mean_7     732
dtype: int64


In [ ]:
# --- Cell 4: Promotion Features (no leakage -- scheduling is known in advance) ---
promo = promotions.copy()
promo['start_date'] = pd.to_datetime(promo['start_date'])
promo['end_date']   = pd.to_datetime(promo['end_date'])

# Vectorized cross-join: every (date, promo) pair, then filter for active ones
promo['_k'] = 1
spine_k = df[['Date']].copy()
spine_k['_k'] = 1
cross = spine_k.merge(promo, on='_k').drop(columns=['_k'])
active = cross[(cross['start_date'] <= cross['Date']) & (cross['end_date'] >= cross['Date'])].copy()

# Aggregate counts per day
promo_agg = active.groupby('Date').agg(
    n_active_promos         = ('promo_id', 'count'),
    n_stackable_promos      = ('stackable_flag', 'sum'),
    promo_category_coverage = ('applicable_category', 'nunique'),
).reset_index()

# Promo type flags
pct_dates = active.loc[active['promo_type'] == 'percentage', 'Date'].unique()
fix_dates = active.loc[active['promo_type'] == 'fixed', 'Date'].unique()
promo_agg['has_percentage_promo'] = promo_agg['Date'].isin(pct_dates).astype(int)
promo_agg['has_fixed_promo']      = promo_agg['Date'].isin(fix_dates).astype(int)

# Max discount percentage among active % promos
max_disc = (active[active['promo_type'] == 'percentage']
            .groupby('Date')['discount_value'].max()
            .reset_index().rename(columns={'discount_value': 'max_discount_pct'}))
promo_agg = promo_agg.merge(max_disc, on='Date', how='left')
promo_agg['max_discount_pct'] = promo_agg['max_discount_pct'].fillna(0.0)

# Days since last promo ended (vectorized via searchsorted on nanoseconds)
all_ends_ns = np.sort(pd.to_datetime(promo['end_date'].unique()).astype(np.int64))
dates_ns    = df['Date'].astype(np.int64).values
idx         = np.searchsorted(all_ends_ns, dates_ns, side='left') - 1
ns_per_day  = 86_400 * 1_000_000_000
days_since  = np.where(idx >= 0,
    (dates_ns - all_ends_ns[np.maximum(idx, 0)]) // ns_per_day, 999)
dsince_df = pd.DataFrame({'Date': df['Date'], 'days_since_last_promo': days_since})

# Merge all promo features into full date spine
promo_full = df[['Date']].merge(promo_agg, on='Date', how='left')
for col in ['n_active_promos', 'n_stackable_promos', 'promo_category_coverage',
            'has_percentage_promo', 'has_fixed_promo']:
    promo_full[col] = promo_full[col].fillna(0).astype(int)
promo_full['max_discount_pct'] = promo_full['max_discount_pct'].fillna(0.0)
promo_full = promo_full.merge(dsince_df, on='Date', how='left')

promo_cols = [c for c in promo_full.columns if c != 'Date']
df = df.merge(promo_full, on='Date', how='left')

# EDA: regime 2019+ — tuong tac promo (calendar regime, khong leakage)
df['promo_n_active_x_post2019'] = df['n_active_promos'].astype(float) * df['post_2019_regime'].astype(float)
df['promo_max_disc_x_post2019'] = df['max_discount_pct'].astype(float) * df['post_2019_regime'].astype(float)
df['promo_stackable_x_post2019'] = df['n_stackable_promos'].astype(float) * df['post_2019_regime'].astype(float)

print(f'Promo features: {len(promo_cols)}')
print(df[['Date', 'n_active_promos', 'max_discount_pct', 'days_since_last_promo']].tail(5))


Promo features: 7
           Date  n_active_promos  max_discount_pct  days_since_last_promo
4376 2024-06-27                0               0.0                    544
4377 2024-06-28                0               0.0                    545
4378 2024-06-29                0               0.0                    546
4379 2024-06-30                0               0.0                    547
4380 2024-07-01                0               0.0                    548


In [ ]:
# --- Cell 5: Web Traffic Features with Seasonal-Naive Fill for Test Period ---
# web_traffic covers 2013-01-01 -> 2022-12-31; test period is 2023-01-01 -> 2024-07-01.
# Strategy: fill test dates with same calendar date from 1 year ago (year-ago copy).
# Fallback: 2 years ago if 1-year-ago value is also NaN.
# After filling, apply shift(1) leakage guard before merging into df.

wt = web_traffic.copy().rename(columns={'date': 'Date'}).sort_values('Date')

# Aggregate all traffic sources per day
wt_agg = wt.groupby('Date').agg(
    sessions_total   = ('sessions', 'sum'),
    unique_visitors  = ('unique_visitors', 'sum'),
    page_views       = ('page_views', 'sum'),
    bounce_rate_mean = ('bounce_rate', 'mean'),
    avg_session_dur  = ('avg_session_duration_sec', 'mean'),
    n_sources        = ('traffic_source', 'nunique'),
).reset_index()

# Per-source session fractions (organic, paid, social, email)
SRC_MAP = {'organic_search': 'organic', 'paid_search': 'paid',
           'social_media': 'social', 'email_campaign': 'email'}
for src_full, src_short in SRC_MAP.items():
    src_data = (wt[wt['traffic_source'] == src_full]
                .groupby('Date')['sessions'].sum()
                .reset_index().rename(columns={'sessions': f'sess_{src_short}'}))
    wt_agg = wt_agg.merge(src_data, on='Date', how='left')
    wt_agg[f'sess_{src_short}'] = wt_agg[f'sess_{src_short}'].fillna(0)
    wt_agg[f'pct_{src_short}']  = wt_agg[f'sess_{src_short}'] / (wt_agg['sessions_total'] + 1e-9)

# feedback-fe: organic search dominance vs paid (phan ra nguon)
wt_agg['organic_minus_paid_pp'] = wt_agg['pct_organic'] - wt_agg['pct_paid']
wt_agg['organic_over_paid_sessions'] = wt_agg['sess_organic'] / (wt_agg['sess_paid'] + 1.0)

traffic_raw_cols = [c for c in wt_agg.columns if c != 'Date']

# Seasonal-naive fill: shift traffic values 1 and 2 years forward
wt_y1 = wt_agg.copy()
wt_y1['Date'] = wt_y1['Date'] + pd.DateOffset(years=1)
wt_y2 = wt_agg.copy()
wt_y2['Date'] = wt_y2['Date'] + pd.DateOffset(years=2)

# Merge actual + year-shifted onto full date spine
wt_full = pd.DataFrame({'Date': pd.date_range(TRAIN_START, TEST_END, freq='D')})
wt_full = wt_full.merge(wt_agg, on='Date', how='left')
wt_full = wt_full.merge(
    wt_y1.rename(columns={c: c + '_y1' for c in traffic_raw_cols}), on='Date', how='left')
wt_full = wt_full.merge(
    wt_y2.rename(columns={c: c + '_y2' for c in traffic_raw_cols}), on='Date', how='left')
for col in traffic_raw_cols:
    wt_full[col] = wt_full[col].fillna(wt_full[col + '_y1']).fillna(wt_full[col + '_y2'])
wt_full.drop(columns=[c for c in wt_full.columns if c.endswith('_y1') or c.endswith('_y2')],
             inplace=True)

# Nonlinear traffic (diminishing returns): log, binh phuong scale, session/visitor
_st = wt_full['sessions_total'].fillna(0).clip(lower=0).astype(float)
_uv = wt_full['unique_visitors'].fillna(0).clip(lower=0).astype(float)
wt_full['sessions_total_log1p'] = np.log1p(_st)
wt_full['sessions_total_sq_scaled'] = (_st ** 2) / 1e12
wt_full['sessions_per_visitor'] = _st / (_uv + 1.0)

traffic_derived = ['sessions_total_log1p', 'sessions_total_sq_scaled', 'sessions_per_visitor']
traffic_for_lag = traffic_raw_cols + traffic_derived

df = df.merge(wt_full, on='Date', how='left')

_obs_dates = set(wt_agg['Date'].dt.normalize().astype('datetime64[ns]'))
df['web_traffic_yoy_imputed'] = (~df['Date'].dt.normalize().isin(_obs_dates)).astype(int)

# Leakage guard: shift all traffic features by 1 day before using as predictors
for col in traffic_for_lag:
    df[f'{col}_lag1'] = df[col].shift(1)
    df[f'{col}_roll7'] = df[col].shift(1).rolling(7, min_periods=3).mean()
df.drop(columns=traffic_for_lag, inplace=True, errors='ignore')

# Derived quality signal: engaged visits = sessions * (1 - bounce_rate)
if 'sessions_total_lag1' in df.columns and 'bounce_rate_mean_lag1' in df.columns:
    df['engaged_visits_lag1'] = df['sessions_total_lag1'] * (1 - df['bounce_rate_mean_lag1'])

wt_feat_cols = [c for c in df.columns
                if any(k in c for k in ['session', 'visitor', 'bounce', 'page_view', 'web_traffic',
                                         'avg_session', 'n_source', 'pct_', 'sess_', 'engaged',
                                         'sq_scaled', 'log1p', 'sessions_per',
                                         'organic_minus', 'organic_over_paid'])]
test_nan = df.loc[~df['is_train'], 'sessions_total_lag1'].isna().sum()
print(f'Web traffic features: {len(wt_feat_cols)}')
print(f'Test NaN in sessions_total_lag1: {test_nan} (target = 0 after seasonal fill)')


Web traffic features: 29
Test NaN in sessions_total_lag1: 1 (target = 0 after seasonal fill)


In [ ]:
# --- Cell 6: Inventory Features with Forward-Fill into Test Period ---
# inventory covers only 2022-10, 2022-11, 2022-12 (3 monthly snapshots).
# Leakage guard: snapshot at end of month M is valid starting month M+1.
# Strategy: merge by year-month, then ffill() to carry last known values into test period.

inv = inventory.copy()

inv_agg = inv.groupby('snapshot_date').agg(
    inv_stockout_pct   = ('stockout_flag', 'mean'),
    inv_overstock_pct  = ('overstock_flag', 'mean'),
    inv_reorder_pct    = ('reorder_flag', 'mean'),
    inv_fill_rate_mean = ('fill_rate', 'mean'),
    inv_sell_through   = ('sell_through_rate', 'mean'),
    inv_dos_mean       = ('days_of_supply', 'mean'),
    inv_stockout_days  = ('stockout_days', 'mean'),
    inv_units_sold     = ('units_sold', 'sum'),
    inv_stock_on_hand  = ('stock_on_hand', 'sum'),
).reset_index()

# snapshot of month M is first available at the beginning of month M+1
inv_agg['valid_from'] = inv_agg['snapshot_date'] + pd.offsets.MonthBegin(1)
inv_agg['ym'] = inv_agg['valid_from'].dt.to_period('M')

inv_cols = [c for c in inv_agg.columns if c.startswith('inv_')]
df['ym'] = df['Date'].dt.to_period('M')
df = df.merge(inv_agg[['ym'] + inv_cols], on='ym', how='left')
df.drop(columns=['ym'], inplace=True)

# Streetwear category inventory (dominant category ~80% revenue from EDA)
sw_inv = inv[inv['category'] == 'Streetwear'].groupby('snapshot_date').agg(
    sw_stockout_pct = ('stockout_flag', 'mean'),
    sw_sell_through = ('sell_through_rate', 'mean'),
    sw_fill_rate    = ('fill_rate', 'mean'),
).reset_index()
sw_inv['valid_from'] = sw_inv['snapshot_date'] + pd.offsets.MonthBegin(1)
sw_inv['ym'] = sw_inv['valid_from'].dt.to_period('M')
sw_cols = ['sw_stockout_pct', 'sw_sell_through', 'sw_fill_rate']
df['ym'] = df['Date'].dt.to_period('M')
df = df.merge(sw_inv[['ym'] + sw_cols], on='ym', how='left')
df.drop(columns=['ym'], inplace=True)

# Forward-fill: carry Dec-2022 snapshot forward through entire test period
all_inv_cols = inv_cols + sw_cols
df[all_inv_cols] = df[all_inv_cols].ffill()

print(f'Inventory features: {len(all_inv_cols)}')
print(f'Train NaN inv_fill_rate_mean (expected -- pre-Nov-2022): '
      f'{df.loc[df["is_train"], "inv_fill_rate_mean"].isna().sum()}')
print(f'Test  NaN inv_fill_rate_mean (should be 0): '
      f'{df.loc[~df["is_train"], "inv_fill_rate_mean"].isna().sum()}')

Inventory features: 12
Train NaN inv_fill_rate_mean (expected -- pre-Nov-2022): 28
Test  NaN inv_fill_rate_mean (should be 0): 0


In [ ]:
# --- Cell 7: Order-Level Daily Aggregates (lag-shifted for leakage safety) ---
oi = order_items.merge(
    orders[['order_id', 'order_date', 'order_status']], on='order_id', how='left'
)
oi['order_date'] = pd.to_datetime(oi['order_date'])
oi = oi.merge(products[['product_id', 'category', 'segment', 'cogs']], on='product_id', how='left')

oi['line_revenue'] = oi['quantity'] * oi['unit_price'] - oi['discount_amount']
oi['line_cogs'] = oi['quantity'] * oi['cogs']
oi['line_margin'] = oi['line_revenue'] - oi['line_cogs']

# feedback-fe: Premium vs non-Premium daily margin-rate gap (EDA: premium margin thap hon)
prem = (
    oi[oi['segment'] == 'Premium']
    .groupby('order_date', as_index=False)
    .agg(rv=('line_revenue', 'sum'), mg=('line_margin', 'sum'))
)
prem['mr_premium'] = prem['mg'] / prem['rv'].clip(lower=1e-9)
nonp = (
    oi[oi['segment'] != 'Premium']
    .groupby('order_date', as_index=False)
    .agg(rv=('line_revenue', 'sum'), mg=('line_margin', 'sum'))
)
nonp['mr_non_premium'] = nonp['mg'] / nonp['rv'].clip(lower=1e-9)
mgap = prem[['order_date', 'mr_premium']].merge(
    nonp[['order_date', 'mr_non_premium']], on='order_date', how='outer'
)
mgap['premium_margin_gap_daily'] = mgap['mr_premium'].fillna(0) - mgap['mr_non_premium'].fillna(0)
mgap = mgap.rename(columns={'order_date': 'Date'})[['Date', 'premium_margin_gap_daily']]

daily_ord = oi.groupby('order_date').agg(
    n_orders               = ('order_id', 'nunique'),
    n_items                = ('quantity', 'sum'),
    promo_order_pct        = ('has_promo', 'mean'),
    avg_discount_amt       = ('discount_amount', 'mean'),
    streetwear_revenue_pct = ('category', lambda x: (x == 'Streetwear').mean()),
    premium_order_pct      = ('segment', lambda x: (x == 'Premium').mean()),
    line_revenue_sum       = ('line_revenue', 'sum'),
).reset_index().rename(columns={'order_date': 'Date'})
daily_ord = daily_ord.sort_values('Date').reset_index(drop=True)

daily_ord['aov_line_approx'] = daily_ord['line_revenue_sum'] / daily_ord['n_orders'].clip(lower=1)
# feedback-fe: promo AOV drag — tuong tac co promo x AOV
daily_ord['promo_x_aov_interaction'] = daily_ord['promo_order_pct'] * daily_ord['aov_line_approx']

daily_ord = daily_ord.merge(mgap, on='Date', how='left')
daily_ord['premium_margin_gap_daily'] = daily_ord['premium_margin_gap_daily'].fillna(0.0)

drop_before_lag = {'line_revenue_sum'}
raw_ord_cols = [c for c in daily_ord.columns if c != 'Date' and c not in drop_before_lag]

for col in raw_ord_cols:
    daily_ord[f'{col}_lag1']  = daily_ord[col].shift(1)
    daily_ord[f'{col}_roll7'] = daily_ord[col].shift(1).rolling(7, min_periods=3).mean()

lag_ord_cols = [c for c in daily_ord.columns if 'lag1' in c or 'roll7' in c]
df = df.merge(daily_ord[['Date'] + lag_ord_cols], on='Date', how='left')

# feedback-fe: conversion proxy = orders / sessions (ca hai deu ngay t-1 sau lag pipeline)
df['cr_proxy_orders_per_session_lag1'] = df['n_orders_lag1'] / (df['sessions_total_lag1'] + 1.0)
df['cr_proxy_orders_per_session_roll7'] = (
    df['cr_proxy_orders_per_session_lag1'].rolling(7, min_periods=3).mean())

n_extra = 2
print(f'Order-level lag/roll: {len(lag_ord_cols)} | CR proxy + roll: {n_extra}')


Order-level features: 12


In [ ]:
# --- Cell 7b: Daily Returns & Reviews (lag-safe) ---
# Tong hop theo ngay su kien; shift(1) de khong dung thong tin cung ngay voi target.

ret_df = returns_tbl.copy()
ret_df['return_date'] = pd.to_datetime(ret_df['return_date'])
ret_daily = ret_df.groupby('return_date').agg(
    n_returns_day   = ('return_id', 'count'),
    return_qty_sum  = ('return_quantity', 'sum'),
    refund_amt_sum  = ('refund_amount', 'sum'),
).reset_index().rename(columns={'return_date': 'Date'})

rev_df = reviews_tbl.copy()
rev_df['review_date'] = pd.to_datetime(rev_df['review_date'])
rev_daily = rev_df.groupby('review_date').agg(
    n_reviews_day = ('review_id', 'count'),
    rating_sum    = ('rating', 'sum'),
    rating_cnt    = ('rating', 'count'),
).reset_index().rename(columns={'review_date': 'Date'})
rev_daily['rating_mean_day'] = rev_daily['rating_sum'] / rev_daily['rating_cnt'].clip(lower=1)
rev_daily.drop(columns=['rating_sum', 'rating_cnt'], inplace=True)

df = df.merge(ret_daily, on='Date', how='left')
df = df.merge(rev_daily, on='Date', how='left')

qual_base = ['n_returns_day', 'return_qty_sum', 'refund_amt_sum', 'n_reviews_day', 'rating_mean_day']
for c in qual_base:
    df[c] = df[c].fillna(0.0)

for c in qual_base:
    df[f'{c}_lag1'] = df[c].shift(1)
    df[f'{c}_roll7'] = df[c].shift(1).rolling(7, min_periods=3).mean()
df.drop(columns=qual_base, inplace=True, errors='ignore')

print(f'Quality (returns/reviews) derived lag/roll cols: {len(qual_base) * 2}')



In [ ]:
# --- Cell 8: Leakage Audit (MANDATORY -- must pass before saving) ---

def leakage_audit(df, target_col='Revenue', date_col='Date'):
    print('=' * 60)
    print('LEAKAGE AUDIT')
    print('=' * 60)
    train_df = df[df['is_train']].copy()
    feat_cols = [c for c in df.columns if c not in [date_col, 'Revenue', 'COGS', 'is_train']]

    # 1. Column name check: Revenue/COGS-derived cols must be lag/roll/ratio/wow/yoy variants
    danger = [
        c for c in feat_cols
        if ('revenue' in c.lower() or 'cogs' in c.lower())
        and not any(k in c.lower() for k in ('lag', 'roll', 'ratio', 'wow', 'yoy', 'expand'))
    ]
    if danger:
        print(f'[WARN] Dangerous column names: {danger}')
    else:
        print('[OK]  Column name check passed')

    # 2. Correlation check: |corr(feature, Revenue)| should not exceed 0.99
    high_corr = []
    for col in feat_cols:
        if train_df[col].nunique() < 2:
            continue
        try:
            corr = train_df[[col, target_col]].dropna().corr().iloc[0, 1]
            if abs(corr) > 0.99:
                high_corr.append((col, round(corr, 4)))
        except Exception:
            pass
    if high_corr:
        print(f'[WARN] |corr| > 0.99 with {target_col}: {high_corr}')
    else:
        print(f'[OK]  No feature has |corr| > 0.99 with {target_col}')

    # 3. Lag-1 consistency: rev_lag_1[t] must equal Revenue[t-1]
    if 'rev_lag_1' in df.columns:
        df_idx = df.set_index(date_col)
        sample = train_df.dropna(subset=['rev_lag_1', 'Revenue']).head(100)
        mismatch = 0
        for _, row in sample.iterrows():
            prev = row[date_col] - pd.Timedelta(days=1)
            if prev in df_idx.index:
                prev_rev = df_idx.loc[prev, 'Revenue']
                if pd.notna(prev_rev) and abs(row['rev_lag_1'] - prev_rev) > 0.01:
                    mismatch += 1
        print(f'[OK]  rev_lag_1 consistency: {mismatch}/100 mismatches (target = 0)')

    print('=' * 60)
    return len(danger) == 0 and len(high_corr) == 0

audit_passed = leakage_audit(df)
assert audit_passed, 'LEAKAGE DETECTED -- investigate before proceeding!'

LEAKAGE AUDIT
[OK]  Column name check passed


[OK]  No feature has |corr| > 0.99 with Revenue
[OK]  rev_lag_1 consistency: 0/100 mismatches (target = 0)


In [ ]:
# --- Cell 9: Handle Missing Values + Split + Save Parquet ---
# Inventory: early train rows (pre-2022-11) are NaN because no snapshot existed.
# Impute with train-set median so GBMs don't have to learn from very sparse signals.
for col in [c for c in df.columns if c.startswith('inv_') or c.startswith('sw_')]:
    med = df.loc[df['is_train'], col].median()
    if pd.notna(med):
        df[col] = df[col].fillna(med)

# fill_rate defaults to 1.0 (fully filled) if still NaN
for col in ['inv_fill_rate_mean', 'sw_fill_rate']:
    if col in df.columns:
        df[col] = df[col].fillna(1.0)

# NaN report (lag/rolling NaN at start of series is expected -- GBMs handle natively)
nan_report = df.isna().sum().sort_values(ascending=False)
print('Top 10 columns with NaN:')
print(nan_report[nan_report > 0].head(10))

# Split and save
train_feat = df[df['is_train']].copy()
test_feat  = df[~df['is_train']].copy()
feat_cols  = [c for c in df.columns if c not in ['Revenue', 'COGS', 'is_train']]

print(f'Train features shape : {train_feat.shape}')
print(f'Test  features shape : {test_feat.shape}')
print(f'Feature count (- Date): {len(feat_cols) - 1}')

train_feat.to_parquet(f'{FEAT_DIR}/train_features.parquet', index=False)
test_feat.to_parquet(f'{FEAT_DIR}/test_features.parquet', index=False)
print(f'[OK] Saved {FEAT_DIR}/train_features.parquet')
print(f'[OK] Saved {FEAT_DIR}/test_features.parquet')

Top 10 columns with NaN:
rev_yoy                  913
rev_ratio_365_730        913
rev_ratio_7_365          906
rev_yoy_growth_90d       882
rev_yoy_growth_180d      852
rev_2yoy_roll_mean_28    743
rev_2yoy_roll_mean_14    736
rev_same_week_2y_mean    735
rev_lag_735              735
rev_2yoy_roll_mean_7     732
dtype: int64


Train features shape : (3837, 161)
Test  features shape : (548, 161)
Feature count (- Date): 157


[OK] Saved ../data/features/train_features.parquet
[OK] Saved ../data/features/test_features.parquet


In [ ]:
# --- Cell 10: Feature Summary + Top-30 Correlation Chart ---
feature_groups = {
    'Calendar':     [c for c in df.columns if any(k in c for k in
                     ('dow', 'month', 'quarter', 'year', 'weekend', 'tet', 'holiday',
                      'regime', 'season', 'school', 'dayof', 'weekof', 'sin_', 'cos_',
                      'peak_may', 'recovery_year'))],
    'Revenue Lag':  [c for c in df.columns if 'rev_lag' in c],
    'Revenue Roll': [c for c in df.columns if any(k in c for k in
                     ('rev_roll', 'rev_ratio', 'rev_wow', 'rev_yoy', 'rev_expand'))],
    'COGS':         [c for c in df.columns if 'cogs' in c],
    'Promotion':    [c for c in df.columns if 'promo' in c or 'days_since_last' in c],
    'Web Traffic':  [c for c in df.columns if any(k in c for k in
                     ('session', 'visitor', 'bounce', 'page_view', 'web_traffic',
                      'avg_session', 'n_source', 'pct_', 'sess_', 'engaged',
                      'sq_scaled', 'log1p', 'sessions_per',
                      'organic_minus', 'organic_over_paid'))],
    'Inventory':    [c for c in df.columns if c.startswith('inv_') or c.startswith('sw_')],
    'Orders':       [c for c in df.columns if any(k in c for k in
                     ('n_order', 'n_item', 'discount', 'streetwear', 'premium_order',
                      'cr_proxy', 'aov_', 'premium_margin', 'promo_x_aov'))],
    'Quality':      [c for c in df.columns if any(k in c for k in
                     ('return', 'refund', 'review', 'rating_mean'))],
}

print('FEATURE ENGINEERING SUMMARY')
print('=' * 45)
total = 0
for group, cols in feature_groups.items():
    print(f'  {group:<16}: {len(cols):>3} features')
    total += len(cols)
print(f'  {"TOTAL":<16}: {total:>3} features')

# Top-30 |Pearson correlation| with Revenue
train_loaded = pd.read_parquet(f'{FEAT_DIR}/train_features.parquet')
feat_cols_plot = [c for c in train_loaded.columns if c not in ('Date', 'Revenue', 'COGS', 'is_train')]
corr_ser = (
    train_loaded[feat_cols_plot + ['Revenue']]
    .corr()['Revenue']
    .drop('Revenue', errors='ignore')
    .abs()
    .sort_values(ascending=False)
    .head(30)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 9))
corr_ser.plot(kind='barh', ax=ax, color='#2563EB', edgecolor='none')
ax.set_title('Top 30 Features by |Pearson Correlation| with Revenue', fontweight='bold')
ax.set_xlabel('|Pearson Correlation|')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'[OK] Chart saved: {CHART_DIR}/feature_correlation.png')


FEATURE ENGINEERING SUMMARY
  Calendar        :  30 features
  Revenue Lag     :  20 features
  Revenue Roll    :  35 features
  COGS            :   6 features
  Promotion       :   8 features
  Web Traffic     :  35 features
  Inventory       :  12 features
  Orders          :  13 features
  TOTAL           : 159 features


[OK] Chart saved: ../reports/charts/fe/feature_correlation.png
